# Tensorflow Object Detection API and AWS Sagemaker - Faster R-CNN ResNet152 V1 640x640

In this notebook, you will train and evaluate different models using the [Tensorflow Object Detection API](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/) and [AWS Sagemaker](https://aws.amazon.com/sagemaker/). 

If you ever feel stuck, you can refer to this [tutorial](https://aws.amazon.com/blogs/machine-learning/training-and-deploying-models-using-tensorflow-2-with-the-object-detection-api-on-amazon-sagemaker/).

## Dataset

We are using the [Waymo Open Dataset](https://waymo.com/open/) for this project. The dataset has already been exported using the tfrecords format. The files have been created following the format described [here](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/training.html#create-tensorflow-records). You can find data stored on [AWS S3](https://aws.amazon.com/s3/), AWS Object Storage. The images are saved with a resolution of 640x640.

In [1]:
%%capture
%pip install tensorflow_io sagemaker==2.* sagemaker-experiments -U

In [19]:
!pip list

Package                      Version
---------------------------- ------------
absl-py                      2.4.0
annotated-doc                0.0.4
annotated-types              0.7.0
antlr4-python3-runtime       4.9.3
anyio                        4.13.0
argon2-cffi                  25.1.0
argon2-cffi-bindings         25.1.0
arrow                        1.4.0
asttokens                    3.0.1
astunparse                   1.6.3
async-lru                    2.3.0
attrs                        25.4.0
Authlib                      1.6.9
autovizwidget                0.23.0
awscli                       1.45.6
awscrt                       0.31.2
babel                        2.18.0
backports.zstd               1.3.0
bcrypt                       5.0.0
beautifulsoup4               4.14.3
bleach                       6.3.0
bokeh                        3.9.0
boto3                        1.43.6
botocore                     1.43.6
Brotli                       1.2.0
cached-property              1.5.2


In [17]:
import os
from sagemaker.debugger import TensorBoardOutputConfig, DebuggerHookConfig, CollectionConfig
import sagemaker
from sagemaker.estimator import Estimator
from framework import CustomFramework

Save the IAM role in a variable called `role`. This would be useful when training the model.

In [3]:
#role = sagemaker.get_execution_role()
role = "arn:aws:iam::261156800081:role/sdc-sagemaker-jose"
print(role)

arn:aws:iam::261156800081:role/sdc-sagemaker-jose


In [4]:
# The train and val paths below are public S3 buckets created by Udacity for this project
inputs = {'train': 's3://cd2688-object-detection-tf2/train/', 
          'val': 's3://cd2688-object-detection-tf2/val/'} 

# Insert path of a folder in your personal S3 bucket to store tensorboard logs.
tensorboard_s3_prefix = 's3://sdc-objdetection-261156800081-us-east-1-an/logs/'

## Container

To train the model, you will first need to build a [docker](https://www.docker.com/) container with all the dependencies required by the TF Object Detection API. The code below does the following:
* clone the Tensorflow models repository
* get the exporter and training scripts from the repository
* build the docker image and push it 
* print the container name

In [5]:
%%bash

# clone the repo and get the scripts
git clone https://github.com/tensorflow/models.git docker/models

# get model_main and exporter_main files from TF2 Object Detection GitHub repository
cp docker/models/research/object_detection/exporter_main_v2.py source_dir 
cp docker/models/research/object_detection/model_main_tf2.py source_dir

Cloning into 'docker/models'...


In [6]:
# build and push the docker image. This code can be commented out after being run once.
# This will take around 10 mins.
image_name = 'tf2-object-detection'
!sh ./docker/build_and_push.sh $image_name 2>&1 | tail -n 500

#14 7.531   Downloading pydot-1.4.2-py2.py3-none-any.whl.metadata (8.0 kB)
#14 7.571 Collecting redis<6,>=5.0.0 (from apache-beam->object-detection==0.1)
#14 7.574   Downloading redis-5.3.1-py3-none-any.whl.metadata (9.2 kB)
#14 7.625 Collecting requests<3.0.0,>=2.24.0 (from apache-beam->object-detection==0.1)
#14 7.629   Downloading requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
#14 7.634 Requirement already satisfied: typing-extensions>=3.7.0 in /usr/local/lib/python3.8/dist-packages (from apache-beam->object-detection==0.1) (4.5.0)
#14 7.704 Collecting zstandard<1,>=0.18.0 (from apache-beam->object-detection==0.1)
#14 7.709   Downloading zstandard-0.23.0-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
#14 7.799 Collecting pyarrow<17.0.0,>=3.0.0 (from apache-beam->object-detection==0.1)
#14 7.805   Downloading pyarrow-16.1.0-cp38-cp38-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
#14 7.817 Collecting pyarrow-hotfix<1 (from apache-beam->object-detection==0

To verify that the image was correctly pushed to the [Elastic Container Registry](https://aws.amazon.com/ecr/), you can look at it in the AWS webapp. For example, below you can see that three different images have been pushed to ECR. You should only see one, called `tf2-object-detection`.
![ECR Example](../data/example_ecr.png)


In [7]:
# display the container name
with open (os.path.join('docker', 'ecr_image_fullname.txt'), 'r') as f:
    container = f.readlines()[0][:-1]

print(container)

261156800081.dkr.ecr.us-east-1.amazonaws.com/tf2-object-detection:20260522031517


## Pre-trained model from model zoo

As often, we are not training from scratch and we will be using a pretrained model from the TF Object Detection model zoo. You can find pretrained checkpoints [here](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md). Because your time is limited for this project, we recommend to only experiment with the following models:
* SSD MobileNet V2 FPNLite 640x640	
* SSD ResNet50 V1 FPN 640x640 (RetinaNet50)	
* Faster R-CNN ResNet50 V1 640x640	
* EfficientDet D1 640x640	
* Faster R-CNN ResNet152 V1 640x640	

In the code below, the EfficientDet D1 model is downloaded and extracted. This code should be adjusted if you were to experiment with other architectures.

In [8]:
%%bash
mkdir /tmp/checkpoint
mkdir source_dir/checkpoint
#wget -O /tmp/efficientdet.tar.gz http://download.tensorflow.org/models/object_detection/tf2/20200711/efficientdet_d1_coco17_tpu-32.tar.gz
#tar -zxvf /tmp/efficientdet.tar.gz --strip-components 2 --directory source_dir/checkpoint efficientdet_d1_coco17_tpu-32/checkpoint

wget -O /tmp/fcnn.tar.gz http://download.tensorflow.org/models/object_detection/tf2/20200711/faster_rcnn_resnet152_v1_640x640_coco17_tpu-8.tar.gz
tar -zxvf /tmp/fcnn.tar.gz --strip-components 2 --directory source_dir/checkpoint faster_rcnn_resnet152_v1_640x640_coco17_tpu-8/checkpoint

--2026-05-22 03:21:38--  http://download.tensorflow.org/models/object_detection/tf2/20200711/faster_rcnn_resnet152_v1_640x640_coco17_tpu-8.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 64.233.180.207, 142.251.163.207, 142.251.167.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|64.233.180.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 470656289 (449M) [application/x-tar]
Saving to: ‘/tmp/fcnn.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 11.0M 41s
    50K .......... .......... .......... .......... ..........  0% 18.5M 33s
   100K .......... .......... .......... .......... ..........  0% 22.2M 28s
   150K .......... .......... .......... .......... ..........  0% 20.8M 27s
   200K .......... .......... .......... .......... ..........  0% 21.9M 25s
   250K .......... .......... .......... .......... ..........  0% 21.3M 25s
   300K .......... .......... .......... .......... ...

faster_rcnn_resnet152_v1_640x640_coco17_tpu-8/checkpoint/ckpt-0.data-00000-of-00001
faster_rcnn_resnet152_v1_640x640_coco17_tpu-8/checkpoint/checkpoint
faster_rcnn_resnet152_v1_640x640_coco17_tpu-8/checkpoint/ckpt-0.index


## Edit pipeline.config file

The [`pipeline.config`](source_dir/pipeline.config) in the `source_dir` folder should be updated when you experiment with different models. The different config files are available [here](https://github.com/tensorflow/models/tree/master/research/object_detection/configs/tf2).

>Note: The provided `pipeline.config` file works well with the `EfficientDet` model. You would need to modify it when working with other models.

## Launch Training Job

Now that we have a dataset, a docker image and some pretrained model weights, we can launch the training job. To do so, we create a [Sagemaker Framework](https://sagemaker.readthedocs.io/en/stable/frameworks/index.html), where we indicate the container name, name of the config file, number of training steps etc.

The `run_training.sh` script does the following:
* train the model for `num_train_steps` 
* evaluate over the val dataset
* export the model

Different metrics will be displayed during the evaluation phase, including the mean average precision. These metrics can be used to quantify your model performances and compare over the different iterations.

You can also monitor the training progress by navigating to **Training -> Training Jobs** from the Amazon Sagemaker dashboard in the Web UI.

In [18]:
tensorboard_output_config = sagemaker.debugger.TensorBoardOutputConfig(
    s3_output_path=tensorboard_s3_prefix,
    container_local_output_path='/opt/training/'
)

# Configure Debugger Hook
hook_config = DebuggerHookConfig(
    s3_output_path=tensorboard_s3_prefix,
    collection_configs=[
        CollectionConfig("loss"),
        CollectionConfig("gradients"),
    ]
)

estimator = CustomFramework(
    role=role,
    image_uri=container,
    entry_point='run_training.sh',
    source_dir='source_dir/',
    hyperparameters={
        "model_dir": "/opt/training",        
        "pipeline_config_path": "faster_rcnn_resnet152_v1_640x640_coco17_tpu-8.config",
        "num_train_steps": "2000",    
        "sample_1_of_n_eval_examples": "1"
    },
    instance_count=1,
    instance_type='ml.g5.xlarge',
    debugger_hook_config=hook_config,
    tensorboard_output_config=tensorboard_output_config,
    disable_profiler=True,
    base_job_name='tf2-object-detection'
)

estimator.fit(inputs)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: tf2-object-detection-2026-05-22-04-35-51-600


2026-05-22 04:36:07 Starting - Starting the training job
2026-05-22 04:36:07 Pending - Training job waiting for capacity............
2026-05-22 04:37:48 Pending - Preparing the instances for training...
2026-05-22 04:38:36 Downloading - Downloading the training image...
2026-05-22 04:39:07 Training - Training image download completed. Training in progress../usr/local/lib/python3.8/dist-packages/paramiko/transport.py:32: CryptographyDeprecationWarning: Python 3.8 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.8.
  from cryptography.hazmat.backends import default_backend
2026-05-22 04:39:07,988 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code  is empty, unzipping
2026-05-22 04:39:11,597 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-22 04:39:11,635 sagemaker-training-toolkit INFO     No Neurons detected (norma

Show tensorboards.

TensorBoard runs on your notebook instance, and you can open it by visiting the URL:
https://your-notebook-instance-name.notebook.your-region.sagemaker.aws/proxy/6006/

In [ ]:
job_artifacts_path = estimator.latest_job_tensorboard_artifacts_path()
tensorboard_s3_output_path = f'{job_artifacts_path}/train'

!F_CPP_MIN_LOG_LEVEL=3 AWS_REGION=us-east-1 tensorboard --logdir=$tensorboard_s3_output_path

/home/ec2-user/anaconda3/envs/tensorflow2_p310/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-22 04:50:43.279202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-22 04:50:43.303157: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-22 04:50:43.303208: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registe

You should be able to see your model training in the AWS webapp as shown below:
![ECR Example](../data/example_trainings.png)


## Improve on the initial model

Most likely, this initial experiment did not yield optimal results. However, you can make multiple changes to the `pipeline.config` file to improve this model. One obvious change consists in improving the data augmentation strategy. The [`preprocessor.proto`](https://github.com/tensorflow/models/blob/master/research/object_detection/protos/preprocessor.proto) file contains the different data augmentation method available in the Tf Object Detection API. Justify your choices of augmentations in the write-up.

Keep in mind that the following are also available:
* experiment with the optimizer: type of optimizer, learning rate, scheduler etc
* experiment with the architecture. The Tf Object Detection API model zoo offers many architectures. Keep in mind that the pipeline.config file is unique for each architecture and you will have to edit it.
* visualize results on the test frames using the `2_deploy_model` notebook available in this repository.

In the cell below, write down all the different approaches you have experimented with, why you have chosen them and what you would have done if you had more time and resources. Justify your choices using the tensorboard visualizations (take screenshots and insert them in your write-up), the metrics on the evaluation set and the generated animation you have created with [this tool](../2_run_inference/2_deploy_model.ipynb).

In [ ]:
# your write-up goes here.